In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [ ]:
X, y = make_classification(
    n_samples=1200, n_features=25,
    n_informative=18, n_redundant=4,
    random_state=7
)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=7, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=7, stratify=y_temp)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

### Task 2: Architecture Design Justification

- **Layer Count**: We use 4 hidden layers of size 128, 64, 32, and 16. This provides a deep representation bottleneck that allows extracting hierarchical patterns from the 18 informative features.
- **Batch Normalization**: Placed immediately after each Dense layer. This normalizes input distributions for the activations, accelerating training stability and preventing gradient issues.
- **Dropout**: A rate of 0.3 is applied after each hidden layer's Batch Normalization. This injects noise during training to prevent the network from over-relying on specific features, improving validation robustness.
- **Adam Optimizer**: Learning rate is configured at 0.001. Adam combines AdaGrad and RMSProp features, adjusting step size dynamically based on first and second moments of the gradient.
- **Output Layer**: A single output unit with a Sigmoid activation mapping real numbers to range [0, 1] for binary probability estimation. Binary crossentropy is the mathematically appropriate loss function.

In [ ]:
model = Sequential([
    Input(shape=(25,)),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(16, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

custom_lr = 0.001
optimizer = Adam(learning_rate=custom_lr)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    verbose=0
)

In [ ]:
y_pred_prob = model.predict(X_test, verbose=0)
y_pred = (y_pred_prob > 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Test Accuracy: {acc:.4f}")
print(f"Test Precision: {prec:.4f}")
print(f"Test Recall: {rec:.4f}")
print(f"Test F1 Score: {f1:.4f}")

In [ ]:
epochs_run = len(history.history['loss'])
best_epoch = np.argmin(history.history['val_loss'])

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
if epochs_run < 200:
    plt.axvline(x=best_epoch, color='r', linestyle='--', label=f'Best Epoch ({best_epoch})')
plt.title('Loss Curves')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
if epochs_run < 200:
    plt.axvline(x=best_epoch, color='r', linestyle='--', label=f'Best Epoch ({best_epoch})')
plt.title('Accuracy Curves')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### Task 5: Design Justification Report

- **Layer Count**: I chose 4 hidden layers to allow the network to extract high-level abstract interactions from the 18 informative input features before committing to classification. This deep architecture solves representation underfitting on complex datasets.
- **Dropout Rate**: I chose a Dropout rate of 0.3 because it acts as an effective regularizer by zeroing out 30% of node activations dynamically. This prevents co-adaptation of features, solving overfitting on the training set.
- **BatchNorm Placement**: I placed Batch Normalization immediately after each Dense layer. This normalizes input scales to downstream layers, stabilizing training dynamics and accelerating convergence.
- **Optimizer Choice**: The Adam optimizer was chosen due to its ability to compute adaptive learning rates for each parameter. This resolves slow training speed issues seen with standard gradient descent.
- **Learning Rate**: I chose an initial learning rate of 0.001 to ensure stable steps at the beginning of optimization. This balance prevents gradient divergence while avoiding stagnant, slow updates.
- **Callbacks**: The EarlyStopping callback (patience 10) stops training when validation loss stops improving, saving training time and preventing overfitting. The ReduceLROnPlateau callback (factor 0.5, patience 5) decreases the learning rate when loss plateaus, enabling the optimizer to find and settle in a finer local minimum.